### Ridge Classifier: Regression Ko Classification Banana

#### 1. Asal Mein Ye Hai Kya? (Core Concept)
Naam mein "Classifier" hai, par andar se ye actually ek **Regression** algorithm hai. 

Jab hum Logistic Regression use karte thay, toh model hume percentage ya probability batata tha (0 se 1 ke beech). Lekin Ridge Classifier ek alag approach leta hai. Ye data ke beech se ek normal straight line (regression line) nikalta hai, aur fir bas us line ke sign (+ ya -) ko dekh kar bata deta hai ki data kis class ka hai.

#### 2. Labels $y \in \{+1, -1\}$ Kyu Chahiye? (The Intuition)
Normal binary classification mein hum apne data ko $0$ (Not Zero) aur $1$ (Zero) label karte hain. 
Lekin Ridge model regression kar raha hai, jiska output koi probability nahi, balki ek real number (jaise $-5.2$, $+3.8$, $-0.1$) hota hai.

Isliye math ko aasan banane ke liye, hum target labels ko $0, 1$ ki jagah **$-1$ aur $+1$** mein convert kar dete hain.
* Agar model ka output positive ($> 0$) aaya, toh wo Class $+1$ hai.
* Agar model ka output negative ($< 0$) aaya, toh wo Class $-1$ hai.

#### 3. The Math & Solver (Least Squares & SVD)
Pichle models mein hum Gradient Descent (iterative approach) use kar rahe thay jahan model dheere-dheere slope ke neeche aata tha. 

Ridge classifier problem ko ek **"Least Squares"** equation ki tarah dekhta hai:
$$y = Xw$$
Hume weights ($w$) nikalne hain. Mathematically, $w$ nikalne ke liye hume $X$ ko udhar bhej kar uska inverse nikalna padta hai. Par real-world data perfect square matrix nahi hota, jiska seedha inverse nikal sake. 
Isliye model ek advanced matrix calculation technique use karta hai jise **SVD (Singular Value Decomposition)** kehte hain. SVD ek formula hai jo kisi bhi ajeeb shape ki matrix ko tod-fod kar uske optimal weights ($w$) ek jhatke mein nikal kar de deta hai. Isko iterative steps nahi lene padte.

#### 4. Regularization: $\alpha$ (Alpha) vs $C$
Lecture ka aakhiri point bohot zaroori hai. Logistic Regression mein Regularization control karne ke liye humne `C` padha tha, jiska penalty se ulta relation tha (Bada $C$ = Kam Penalty).

Lekin Ridge Classifier mein `alpha` ($\alpha$) parameter use hota hai, jiska direct relation hota hai:
* Bada $\alpha$ = Badi Penalty (Strong L2 Regularization).
* Chota $\alpha$ = Choti Penalty.

Isliye, agar hume bilkul "pure" model chahiye jisme koi penalty na ho (No Regularization), toh hum equation mein **`alpha = 0`** set kar dete hain.

---

In [1]:
import numpy as np
from pprint import pprint

from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline , Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import (
    SGDClassifier , RidgeClassifier , LogisticRegression , LogisticRegressionCV
)
from sklearn.metrics import (
    log_loss,
    ConfusionMatrixDisplay,
    precision_score , recall_score , classification_report,
    precision_recall_curve , roc_curve , roc_auc_score
)

from sklearn.model_selection import (
    cross_validate,
    RandomizedSearchCV,
    GridSearchCV                 
)

from scipy.stats import loguniform

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns


In [15]:
from sklearn.datasets import fetch_openml
x_pd , y_pd = fetch_openml('mnist_784' , version=1 , return_X_y=True)
x = x_pd.to_numpy()
y = y_pd.to_numpy()

scaler = MinMaxScaler()
X = scaler.fit_transform(x)

target_names = np.unique(y)

x_train, x_test, y_train, y_test = X[ :60000], X[60000 : ] , y[ : 60000], y[ 60000 : ]

y_train_0 = np.zeros(len(y_train))
y_test_0 = np.zeros(len(y_test))

index_0 = np.where(y_train=='0')
y_train_0[index_0] = 1

index_0 = np.where(y_test=='0')
y_test_0[index_0] = 1

In [7]:
ridge_clf = RidgeClassifier(
    alpha=0,
)

pipe_ridge = Pipeline([
    ("min_max_scaler" , MinMaxScaler()),
    ("ridge_clf" , ridge_clf)
])

pipe_ridge.fit(x_train , y_train_0)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('min_max_scaler', ...), ('ridge_clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data toprovided `feature_range`.Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the `feature_range`, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data... versionadded:: 0.24",False
,"alpha alpha: float, default=1.0Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`.",0
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If set to false, nointercept will be used in calculations (e.g. data is expected to bealready centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.The default value is determined by scipy.sparse.linalg.",None


In [9]:
y_hat_test_0 = pipe_ridge.predict(x_test)
print(classification_report(y_test_0 , y_hat_test_0))

              precision    recall  f1-score   support

         0.0       0.99      1.00      0.99      9020
         1.0       0.95      0.88      0.92       980

    accuracy                           0.98     10000
   macro avg       0.97      0.94      0.95     10000
weighted avg       0.98      0.98      0.98     10000



In [13]:
pipe_logit = make_pipeline(
    MinMaxScaler(),
    LogisticRegression(
        random_state=1729,
        solver='lbfgs',
        penalty=None
    )   
)

pipe_logit.fit(x_train , y_train_0)

c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('minmaxscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data toprovided `feature_range`.Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the `feature_range`, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data... versionadded:: 0.24",False
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",None
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=

In [20]:
cv_bin_ridge_clf = cross_validate(
    pipe_logit,x_train , y_train_0,cv=5,
    scoring=["precision","recall","f1"],
    return_estimator=True,
    return_train_score=True
)

pprint(cv_bin_ridge_clf)

c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was depr

{'estimator': [Pipeline(steps=[('minmaxscaler', MinMaxScaler()),
                ('logisticregression',
                 LogisticRegression(penalty=None, random_state=1729))]),
               Pipeline(steps=[('minmaxscaler', MinMaxScaler()),
                ('logisticregression',
                 LogisticRegression(penalty=None, random_state=1729))]),
               Pipeline(steps=[('minmaxscaler', MinMaxScaler()),
                ('logisticregression',
                 LogisticRegression(penalty=None, random_state=1729))]),
               Pipeline(steps=[('minmaxscaler', MinMaxScaler()),
                ('logisticregression',
                 LogisticRegression(penalty=None, random_state=1729))]),
               Pipeline(steps=[('minmaxscaler', MinMaxScaler()),
                ('logisticregression',
                 LogisticRegression(penalty=None, random_state=1729))])],
 'fit_time': array([ 4.42135549, 15.04983044, 10.83371544,  6.30831265,  7.09355807]),
 'score_time': array([0.300

In [ ]:
best_estimator_id = np.argmax(cv_bin_ridge_clf['train_f1']); 
print(best_estimator_id)

2


In [23]:
best_estimator = cv_bin_ridge_clf['estimator'][best_estimator_id]

In [27]:
y_hat_test_0 = best_estimator.predict(x_test)
print(classification_report(y_test_0 , y_hat_test_0))

              precision    recall  f1-score   support

         0.0       1.00      0.99      1.00      9020
         1.0       0.95      0.97      0.96       980

    accuracy                           0.99     10000
   macro avg       0.97      0.98      0.98     10000
weighted avg       0.99      0.99      0.99     10000

